In [ ]:
# 1
import os
import requests
from dotenv import load_dotenv

load_dotenv()
STOCK_API_KEY = os.getenv('STOCK_API_KEY', '')

BASE_URL = 'http://apis.data.go.kr/1160100/service/GetStockSecuritiesInfoService/getStockPriceInfo'

class OpenAPIKeyError(Exception): ...
class OpenAPIQuotaError(Exception): ...
class OpenAPIParamError(Exception): ...
class OpenAPIError(Exception): ...

def fetch_stock(page=1, size=5):
    params = {
        'serviceKey': STOCK_API_KEY,
        'resultType': 'json',
        'numOfRows': size,
        'pageNo': page,
        'itmsNm': '삼성전자'
    }

    res = requests.get(BASE_URL, params=params)
    res.raise_for_status() 

    data = res.json()

    if data.get('OpenAPI_ServiceResponse'):
        msg = data.get('OpenAPI_ServiceResponse')['cmmMsgHeader']['errMsg']
        
        if msg == 'SERVICE_KEY_IS_NOT_REGISTERED_ERROR':
            raise OpenAPIKeyError("인증키 오류: 등록되지 않은 키입니다.")
        elif msg == 'LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS_ERROR':
            raise OpenAPIQuotaError("일일 쿼터 초과: 내일 다시 시도하세요.")
        elif msg == 'INVALID_REQUEST_PARAMETER_ERROR':
            raise OpenAPIParamError("필수 파라미터 누락.")
        else:
            raise OpenAPIError(msg)

    header = data['response']['header']
    if header['resultCode'] != '00':
        code = header['resultCode']
        msg = header['resultMsg']
        
        if code == '30' or "SERVICE_KEY_IS_NOT_REGISTERED_ERROR" in msg:
            raise OpenAPIKeyError("인증키 오류")
        elif code == '22' or "LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS_ERROR" in msg:
            raise OpenAPIQuotaError("일일 쿼터 초과")
        elif code == '10' or "INVALID_REQUEST_PARAMETER_ERROR" in msg:
            raise OpenAPIParamError("필수 파라미터 누락")
        else:
            raise OpenAPIError(f"[{code}] {msg}")

    return data['response']['body']['items']['item']

def main():
    try:
        items = fetch_stock(page=1, size=5)
        
        print(f"조회 건수: {len(items)}건")
        for item in items:
            print(item['basDt'], item['itmsNm'], item['clpr'])
            
    except Exception as e:
        print("수집 실패:", e)

if __name__ == '__main__':
    main()

조회 건수: 5건
20260826 삼성전자 261500
20260825 삼성전자 257000
20260824 삼성전자 257000
20260821 삼성전자 281500
20260820 삼성전자 271000


In [ ]:
# 2
import os
import time
import json
import hashlib
import pymysql
import requests
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()
STOCK_API_KEY = os.getenv('STOCK_API_KEY', '')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')

BASE_URL = 'http://apis.data.go.kr/1160100/service/GetStockSecuritiesInfoService/getStockPriceInfo'
DELAY = 0.3
BATCH = 500

class OpenAPIKeyError(Exception): ...
class OpenAPIQuotaError(Exception): ...
class OpenAPIParamError(Exception): ...
class OpenAPIError(Exception): ...

def fetch_stock(page=1, size=5, **kwargs):
    params = {
        'serviceKey': STOCK_API_KEY,
        'resultType': 'json',
        'numOfRows': size,
        'pageNo': page,
        **kwargs  
    }
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    res = requests.get(BASE_URL, params=params, headers=headers)
    res.raise_for_status() 

    data = res.json()
    
    if data.get('OpenAPI_ServiceResponse'):
        msg = data['OpenAPI_ServiceResponse']['cmmMsgHeader']['errMsg']
        raise OpenAPIError(msg)
        
    header = data.get('response', {}).get('header', {})
    if header and header.get('resultCode') != '00':
        raise OpenAPIError(header.get('resultMsg'))

    return res.url, data['response']['body']['items']['item']

def connect_db():
    return pymysql.connect(
        host='localhost',
        user='root',           
        password=DB_PASSWORD,   
        database='fsc_db',
        charset='utf8mb4'
    )

def main():
    CODES = [
        "005930", "000660", "035420", "051910", "005380", 
        "006400", "035720", "068270", "105560", "055550"
    ]
    
    SOURCE = 'fsc_api'
    now = datetime.now()
    rows = []
    
    for code in CODES:
        try:
            url, items = fetch_stock(
                size=300, 
                likeSrtnCd=code, 
                beginBasDt="20250101", 
                endBasDt="20251231"
            )
            
            for item in items:
                payload = json.dumps(item, ensure_ascii=False) 
                
                key_src = f"{SOURCE}|{item['basDt']}|{item['srtnCd']}"
                content_hash = hashlib.sha256(key_src.encode()).hexdigest()
                
                rows.append((SOURCE, url, now, payload, content_hash))
                
            print(f"[{code}] 수집 완료")
            time.sleep(DELAY) 
            
        except Exception as e:
            print(f"[{code}] 수집 실패: {e}")

    if rows:
        conn = connect_db()
        try:
            with conn.cursor() as cur:
                INSERT_SQL = """
                    INSERT INTO raw_item(source, url, collected_at, payload, content_hash)
                    VALUES (%s, %s, %s, %s, %s)
                    ON DUPLICATE KEY UPDATE
                        payload = VALUES(payload),
                        collected_at = NOW()
                """
                for i in range(0, len(rows), BATCH):
                    cur.executemany(INSERT_SQL, rows[i:i + BATCH]) 
                conn.commit()
                print(f"\n총 {len(rows)}건의 데이터 DB 적재")
        except Exception as e:
            print(f"DB 적재 실패: {e}")
            conn.rollback()
        finally:
            conn.close()

if __name__ == '__main__':
    main()

'''
MariaDB [fsc_db]> SELECT source, COUNT(*) AS cnt FROM raw_item GROUP BY source;
+---------+------+
| source  | cnt  |
+---------+------+
| fsc_api | 2420 |
+---------+------+
1 row in set (0.002 sec)
'''

[005930] 수집 완료
[000660] 수집 완료
[035420] 수집 완료
[051910] 수집 완료
[005380] 수집 완료
[006400] 수집 완료
[035720] 수집 완료
[068270] 수집 완료
[105560] 수집 완료
[055550] 수집 완료

총 2420건의 데이터 DB 적재
